# RAG — Course Transcript Search
Chunk + embed the "Production RAG" course transcript so you can search the
lecture content by meaning. Each chunk carries an **approximate video
timestamp**, so a hit tells you where in the 7h38m video to jump to.

- Source: `research/youtube/mHxLXzYjQRE/transcript.md` (395,617 chars, one line)
- Collection: `rag_course_transcript` (separate from `davinci_manual`)
- Embedder: nomic-embed-text (same as the DaVinci pipeline)


## 1. Environment — Load `.env` Config
Reads LLM, embedding, and index settings from `.env`.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    dotenv_file = path / ".env"
    if dotenv_file.exists():
        load_dotenv(dotenv_file, override=True)
        print(f"✅ Loaded .env from {dotenv_file}")
        break

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", "")
EMBED_MODEL = os.getenv("EMBED_MODEL", "")
INDEX_DIR = os.getenv("INDEX_DIR", "index_storage")


✅ Loaded .env from /Users/Shared/.vscode/opencampus/lecture-from-llms-agents/final-project/.env


## 2. Models — Embedder (Ollama) & LLM (vLLM)
Sets up `nomic-embed-text` for embeddings and the remote Qwen LLM (optional — retrieval works without it).

In [3]:
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding

embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=EMBED_BASE_URL.replace("/v1", ""),
)
Settings.embed_model = embed_model

# LLM is optional here — retrieval works without it.
llm = None
if LLM_BASE_URL and LLM_MODEL and LLM_API_KEY:
    from llama_index.llms.openai import OpenAI
    from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS
    ALL_AVAILABLE_MODELS[LLM_MODEL] = 131072
    llm = OpenAI(model=LLM_MODEL, api_base=LLM_BASE_URL, api_key=LLM_API_KEY, max_tokens=4096)
    Settings.llm = llm
    print(f"✅ LLM: {LLM_MODEL} at {LLM_BASE_URL}")
else:
    print("⚠️  LLM not configured — retrieval-only mode")

print(f"✅ Embeddings: {EMBED_MODEL} at {EMBED_BASE_URL}")


✅ LLM: qwen3.8:27b at https://vllm.srv-prod-7.studio.thdi.cc/v1
✅ Embeddings: nomic-embed-text at http://localhost:11434/v1


## 3. Load Transcript — Strip Description & Quality Gate
Reads the raw transcript, removes the YouTube description block, and rejects garbage before chunking.

In [7]:
import re

TRANSCRIPT_PATH = Path("/Users/Shared/.vscode/opencampus/lecture-from-llms-agents/research/youtube/mHxLXzYjQRE/transcript.md")
VIDEO_ID = "mHxLXzYjQRE"
VIDEO_DURATION_SEC = 7*3600 + 38*60 + 38  # 7:38:38

raw = TRANSCRIPT_PATH.read_text(encoding="utf-8")
print(f"📄 Raw transcript: {len(raw)} chars")

# Strip the YouTube description block (everything before the first '>>' speaker marker)
cut = raw.find(">>")
if cut != -1:
    raw = raw[cut+2:].lstrip()
    print(f"✂️  Stripped description block ({cut} chars)")

total_chars = len(raw)
alpha_ratio = sum(c.isalpha() for c in raw) / max(total_chars, 1)
print(f"📄 Spoken transcript: {total_chars} chars | Alpha ratio: {alpha_ratio:.1%}")
if alpha_ratio < 0.30:
    raise ValueError(f"Quality gate FAILED: alpha ratio {alpha_ratio:.1%} < 30%")
print("✅ Quality gate passed")


📄 Raw transcript: 395617 chars
✂️  Stripped description block (438 chars)
📄 Spoken transcript: 395176 chars | Alpha ratio: 78.3%
✅ Quality gate passed


## 4. Chunking — SentenceSplitter + Video Timestamps
Splits the transcript into ~1000-token chunks and attaches an approximate video timestamp to each one.

In [8]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1000"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "200"))
TOP_K = int(os.getenv("TOP_K", "3"))

def fmt_ts(sec):
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

doc = Document(
    text=raw,
    metadata={"source": "rag_course_transcript", "video_id": VIDEO_ID, "file_name": TRANSCRIPT_PATH.name},
)

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents([doc])

# Attach an approximate video timestamp to each chunk (char offset -> time).
# Sequential search keeps positions monotonic despite chunk overlap.
total = len(raw)
search_from = 0
for i, node in enumerate(nodes):
    start = raw.find(node.text[:60], search_from)
    if start == -1:
        start = int((i / len(nodes)) * total)  # fallback: even spacing
    node.metadata["char_start"] = start
    node.metadata["approx_timestamp"] = fmt_ts((start / total) * VIDEO_DURATION_SEC)
    search_from = start + 1

max_chars = max(len(n.text) for n in nodes)
print(f"🧩 Chunks: {len(nodes)} | max {max_chars} chars (~{max_chars//4} tokens)")
print(f"📏 Chunk size: {CHUNK_SIZE} tokens, overlap {CHUNK_OVERLAP}, top_k {TOP_K}")
print("Sample chunks:")
for n in nodes[:3]:
    print(f"  [{n.metadata['approx_timestamp']}] {n.text[:70].replace(chr(10),' ')}...")


🧩 Chunks: 120 | max 4714 chars (~1178 tokens)
📏 Chunk size: 1000 tokens, overlap 200, top_k 3
Sample chunks:
  [00:00:00] so, you follow a rag tutorial. it worked on 10 documents, but then you...
  [00:03:41] and in this case the question is going to go through what we call the ...
  [00:07:59] this is how you do rag with sources. the reason why sources matter is ...


## 5. Embedding & Index — Chroma (`rag_course_transcript`)
Embeds all chunks and persists the index to disk. **Runs once** — re-running deletes and rebuilds the collection.

In [10]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

COLLECTION = "rag_course_transcript"

chroma_client = chromadb.PersistentClient(path=INDEX_DIR)
try:
    chroma_client.delete_collection(COLLECTION)
    print(f"🗑️  Deleted old '{COLLECTION}' collection")
except Exception:
    pass
chroma_collection = chroma_client.get_or_create_collection(COLLECTION)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("🔢 Embedding & building index...")
# NOTE: no from_nodes() in LlamaIndex 0.14 — pass nodes to the constructor.
index = VectorStoreIndex(
    nodes,
    embed_model=embed_model,
    storage_context=storage_context,
    show_progress=True,
)
print(f"✅ Index saved to {INDEX_DIR}/ (collection: {COLLECTION})")

🗑️  Deleted old 'rag_course_transcript' collection
🔢 Embedding & building index...


Generating embeddings:   0%|          | 0/120 [00:00<?, ?it/s]

✅ Index saved to /Users/Shared/.vscode/opencampus/lecture-from-llms-agents/final-project/index_storage/ (collection: rag_course_transcript)


## 6. Retrieval Test — Vector Search + RAG Answer
Test queries with pure vector search (shows timestamps + scores), plus an optional LLM-generated answer.

In [11]:
# --- Retrieval test: pure vector search (works without the LLM) ---
def search(query, k=TOP_K):
    results = index.as_retriever(similarity_top_k=k).retrieve(query)
    print()
    print(f"❓ {query}")
    for i, node in enumerate(results, 1):
        ts = node.metadata.get("approx_timestamp", "?")
        preview = node.text[:140].replace(chr(10), " ")
        print(f"  [{i}] @{ts} score={node.score:.3f} — {preview}...")
    return results

search("How does hybrid search combine vector and keyword results?")
search("What are the main RAG failure modes?")

# --- Optional: full RAG answer (only if the LLM is reachable) ---
if llm:
    query_engine = index.as_query_engine(similarity_top_k=TOP_K, llm=llm)
    resp = query_engine.query("Summarize the production RAG checklist in 3 bullets.")
    print()
    print("💡 " + resp.response)
else:
    print()
    print("⚠️  LLM not configured — skipping generated answer (retrieval only).")



❓ How does hybrid search combine vector and keyword results?
  [1] @02:00:35 score=0.518 — it's going to be ranked one, document 7, rank two and so forth. and then the other side is going to be bm25 search which will be based on ke...
  [2] @02:08:05 score=0.456 — so what are we trying to do is to get to the middle with these tuning weights which is going to be balanced good for mixed query types which...
  [3] @02:12:01 score=0.450 — bm25 matches exact terms and hybrid gets you the best of both worlds. but the fusion math means results aren't always a simple merge of the ...

❓ What are the main RAG failure modes?
  [1] @02:23:22 score=0.353 — that's a clear path. with large launch model, it's different because now we have bad answer. you don't know who created or where this goes, ...
  [2] @04:37:24 score=0.343 — and then we have the clean function here. this method is going to remove dangerous delimiters like triple dashes which attackers love to use...
  [3] @04:33:41 score=0.338 

## 7. Reranking — LLM-as-Reranker
Stage 2 of retrieval: pull top-10 candidates, let the LLM score each one 0–10 for relevance, keep the top-3. Fixes *mis-ordered* results (the right chunk is in the top-10 but ranked too low).

In [14]:
import json
import time
from llama_index.core.llms import ChatMessage

RERANK_TOP_N = 10  # candidates pulled by vector search before reranking

def _strip_think(text):
    """Remove <think>...</think> blocks (Qwen thinking mode) before parsing."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def llm_rerank(query, top_n=RERANK_TOP_N, keep=TOP_K, candidates=None):
    """Score candidates with the LLM and return the top `keep`.

    candidates: optional pre-fetched list (e.g. from hybrid retrieval).
                If None, falls back to vector top_n.
    """
    if llm is None:
        raise RuntimeError("LLM not configured — reranking requires the LLM")
    if candidates is None:
        candidates = index.as_retriever(similarity_top_k=top_n).retrieve(query)

    # Numbered list of candidates (truncated to keep the prompt small)
    snippets = []
    for i, node in enumerate(candidates, 1):
        ts = node.metadata.get("approx_timestamp", "?")
        snippets.append(f"[{i}] (video @{ts})\n{node.text[:600]}")
    numbered = "\n\n".join(snippets)

    prompt = (
        "You are a search reranker. Given a query and numbered text passages, "
        "score each passage 0-10 for how well it ANSWERS the query.\n"
        "10 = directly and fully answers, 5 = partially related, 0 = irrelevant.\n"
        "Return ONLY a JSON object mapping passage number to score, e.g. {\"1\": 8, \"2\": 3}.\n\n"
        f"QUERY: {query}\n\nPASSAGES:\n{numbered}"
    )

    t0 = time.time()
    resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
    elapsed = time.time() - t0

    text = _strip_think(resp.message.content)
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    scores = json.loads(m.group(0)) if m else {}

    scored = []
    for i, node in enumerate(candidates, 1):
        try:
            s = float(scores.get(str(i), 0))
        except (ValueError, TypeError):
            s = 0.0
        node.metadata["rerank_score"] = s
        scored.append(node)
    scored.sort(key=lambda n: n.metadata["rerank_score"], reverse=True)
    print(f"   ⏱️  LLM rerank: {elapsed:.1f}s over {len(candidates)} candidates")
    return scored[:keep]

## 8. Before/After Comparison — Vector Only vs. Reranked
Runs both test queries through plain vector search and through the reranker, side by side. The "failure modes" query is the canary — it retrieves garbage without reranking.

In [13]:
def show(results):
    for i, node in enumerate(results, 1):
        ts = node.metadata.get("approx_timestamp", "?")
        rr = node.metadata.get("rerank_score")
        rr_s = f" rerank={rr:.0f}" if rr is not None else ""
        preview = node.text[:110].replace("\n", " ")
        print(f"  [{i}] @{ts}{rr_s} — {preview}...")

TEST_QUERIES = [
    "How does hybrid search combine vector and keyword results?",
    "What are the main RAG failure modes?",
]

for q in TEST_QUERIES:
    print("=" * 72)
    print(f"❓ {q}")
    print("-" * 72)
    print("BEFORE (vector only, top 3):")
    show(index.as_retriever(similarity_top_k=TOP_K).retrieve(q))
    print("AFTER (vector top 10 → LLM rerank → top 3):")
    show(llm_rerank(q))
    print()

❓ How does hybrid search combine vector and keyword results?
------------------------------------------------------------------------
BEFORE (vector only, top 3):
  [1] @02:00:35 — it's going to be ranked one, document 7, rank two and so forth. and then the other side is going to be bm25 se...
  [2] @02:08:05 — so what are we trying to do is to get to the middle with these tuning weights which is going to be balanced go...
  [3] @02:12:01 — bm25 matches exact terms and hybrid gets you the best of both worlds. but the fusion math means results aren't...
AFTER (vector top 10 → LLM rerank → top 3):
   ⏱️  LLM rerank: 0.9s over 10 candidates
  [1] @02:00:35 rerank=10 — it's going to be ranked one, document 7, rank two and so forth. and then the other side is going to be bm25 se...
  [2] @02:12:01 rerank=9 — bm25 matches exact terms and hybrid gets you the best of both worlds. but the fusion math means results aren't...
  [3] @02:08:05 rerank=6 — so what are we trying to do is to get to the

## 9. Findings — Recall vs. Precision (the core lesson)
**Query 1 (hybrid search)** — vector top-3 was *relevant but misordered*: the RRF chunk was #10 in the pool, tuning-weights chunk was #1. Reranking fixed the **precision**: RRF → 10, tuning → 9, "best of both worlds" → 6.

**Query 2 (failure modes)** — vector top-3 was *irrelevant* (LLM hallucination, prompt injection, config loading). The real failure-modes content (@01:48, @01:55) was **not in the top-10 at all**. The reranker did the right thing: it scored the best available candidate only **3/10** — an honest "I don't have the answer" instead of a confident wrong one. But it *couldn't* fix what retrieval missed.

**The two-stage rule:**
- **Retrieval** = *recall* — is the right chunk in the candidate pool at all?
- **Reranking** = *precision* — is it ordered well?
- A reranker can only reorder what the retriever hands it. **Pipeline order: hybrid retrieval first, reranking second.**

**Next step:** hybrid search (BM25 + vector + RRF) — the keyword champion catches the literal phrase "failure modes" that vector search missed.

### ✅ Resolution (section 11 results)
Hybrid search fixed the canary:
- **BM25 alone** found the failure-modes chunks by exact phrase match ("failure case number two is acronyms and abbreviations" @01:56:45) — vector search had ranked them outside the top-10.
- **Hybrid (RRF)** pulled them into the candidate pool.
- **Hybrid → rerank** then did its job: @01:56:45 scored **8/10** (was 3/10 with vector-only candidates), @00:59:21 (bad-embeddings failure) 7/10.
- Query 1 stayed strong: RRF chunk 9/10, "best of both worlds" 8/10.

**Measured pipeline order, confirmed:** `hybrid retrieval (vector + BM25 + RRF)` → `LLM rerank` → `answer`. Each stage fixed a failure the previous one couldn't.

## 10. Hybrid Search — BM25 + Vector + RRF
Stage 1, upgraded: run vector search **and** BM25 keyword search in parallel, merge with **Reciprocal Rank Fusion** (RRF). Documents that rank well in *both* lists bubble to the top. Fixes the *recall* problem the canary exposed — exact-phrase chunks that vector search misses.

**Balancing the two retrievers** (instructor, video @02:07:35): *"if the weights are closer to the vector, 0.3/0.7, this is going to be semantic heavy. And if it's 0.7/0.3, this is going to be heavy on codes, IDs and so forth, closer to BM25. So what are we trying to do is to get to the middle with these tuning weights, which is going to be balanced, good for mixed query types."* His practical tip (@02:14:09): *"start at 50/50 when we are tuning our weights, and you can adjust based on query patterns."* → We start at **0.5/0.5**.

In [15]:
from rank_bm25 import BM25Okapi

# --- BM25 index over the SAME 120 chunks (in-memory, rebuilt on every run) ---
# NOTE: BM25 has no incremental updates — new docs mean a rebuild.
# Fine here (120 chunks, ~ms); at scale you'd persist it (e.g. Elasticsearch).
import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)
from nltk.tokenize import word_tokenize

corpus_tokens = [word_tokenize(n.text.lower()) for n in nodes]
bm25 = BM25Okapi(corpus_tokens)
print(f"🔤 BM25 index built over {len(nodes)} chunks")

# --- Reciprocal Rank Fusion (the course's merge algorithm) ---
# score(doc) = w_vec / (k + rank_vec) + w_bm25 / (k + rank_bm25)
# A doc ranked high in BOTH lists gets both terms -> bubbles to the top.
RRF_K = 60  # standard constant; dampens the impact of rank position

def rrf_fuse(vec_results, bm25_results, w_vec=0.5, w_bm25=0.5, k=RRF_K, top_n=RERANK_TOP_N):
    """Merge two ranked lists (of nodes) via weighted RRF. Returns top_n nodes."""
    scores = {}
    node_by_id = {}
    for rank, node in enumerate(vec_results):
        nid = node.node_id
        scores[nid] = scores.get(nid, 0.0) + w_vec / (k + rank + 1)
        node_by_id[nid] = node
    for rank, node in enumerate(bm25_results):
        nid = node.node_id
        scores[nid] = scores.get(nid, 0.0) + w_bm25 / (k + rank + 1)
        node_by_id[nid] = node
    ranked = sorted(scores, key=scores.get, reverse=True)[:top_n]
    out = []
    for nid in ranked:
        node = node_by_id[nid]
        node.metadata["rrf_score"] = scores[nid]
        out.append(node)
    return out

def bm25_search(query, top_n=RERANK_TOP_N):
    """Keyword search: score all chunks, return top_n as nodes."""
    scores = bm25.get_scores(word_tokenize(query.lower()))
    top_ids = scores.argsort()[::-1][:top_n]
    return [nodes[i] for i in top_ids]

def hybrid_search(query, top_n=RERANK_TOP_N, w_vec=0.5, w_bm25=0.5):
    """Vector + BM25 in parallel, merged with RRF. Returns top_n candidates."""
    vec = index.as_retriever(similarity_top_k=top_n).retrieve(query)
    kw = bm25_search(query, top_n=top_n)
    return rrf_fuse(vec, kw, w_vec=w_vec, w_bm25=w_bm25, top_n=top_n)

print("✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5)")

🔤 BM25 index built over 120 chunks
✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5)


## 11. Full Pipeline Test — Vector / BM25 / Hybrid / Hybrid+Rerank
All four retrieval strategies side by side for both test queries. The canary ("failure modes") should now find the @01:48/@01:55 chunks via BM25 — and the reranker should finally have a real answer to promote.

In [ ]:
def show4(results):
    for i, node in enumerate(results, 1):
        ts = node.metadata.get("approx_timestamp", "?")
        extra = ""
        for key in ("rerank_score", "rrf_score"):
            if key in node.metadata:
                extra += f" {key.split('_')[0]}={node.metadata[key]:.2f}"
        preview = node.text[:100].replace("\n", " ")
        print(f"  [{i}] @{ts}{extra} — {preview}...")

for q in TEST_QUERIES:
    print("=" * 72)
    print(f"❓ {q}")
    print("-" * 72)
    print("A) VECTOR only (top 3):")
    show4(index.as_retriever(similarity_top_k=TOP_K).retrieve(q))
    print("B) BM25 only (top 3):")
    show4(bm25_search(q, top_n=TOP_K))
    print("C) HYBRID (vector+BM25, RRF, top 3):")
    show4(hybrid_search(q, top_n=TOP_K))
    print("D) HYBRID top 10 → LLM rerank → top 3  (full pipeline):")
    show4(llm_rerank(q, candidates=hybrid_search(q, top_n=RERANK_TOP_N)))
    print()

❓ How does hybrid search combine vector and keyword results?
------------------------------------------------------------------------
A) VECTOR only (top 3):
  [1] @02:00:35 — it's going to be ranked one, document 7, rank two and so forth. and then the other side is going to ...
  [2] @02:08:05 — so what are we trying to do is to get to the middle with these tuning weights which is going to be b...
  [3] @02:12:01 — bm25 matches exact terms and hybrid gets you the best of both worlds. but the fusion math means resu...
B) BM25 only (top 3):
  [1] @02:00:35 — it's going to be ranked one, document 7, rank two and so forth. and then the other side is going to ...
  [2] @02:08:05 — so what are we trying to do is to get to the middle with these tuning weights which is going to be b...
  [3] @01:56:45 — well, it's going to return documents about specifications and products, but not the one with the ske...
C) HYBRID (vector+BM25, RRF, top 3):
  [1] @02:00:35 rrf=0.02 — it's going to be ranked 

## 12. Citations & Grounding — Closing Failure Mode #5 (Hallucination)
The course's defense (@00:06:50): *"to make sure the results are grounded in the retrieved documents, add sources to each chunk."* We don't modify the chunks — they already carry `approx_timestamp` in metadata. The enforcement happens in the **prompt**:
1. **Numbered context** — each chunk labeled `[n] (video @ts)` so the LLM has concrete things to cite
2. **Grounding instruction** — answer ONLY from the context, cite inline, admit when it's not there
3. **Refusal threshold** — top rerank score < 5/10 → "not found in source" (the reranker's score *is* the confidence signal)

In [17]:
REFUSAL_THRESHOLD = 5.0  # top rerank score below this -> "not found in source"

def answer_with_citations(query, top_n=RERANK_TOP_N, keep=TOP_K):
    """Full pipeline with grounding + citations + refusal.

    hybrid retrieval -> LLM rerank -> (refuse if low confidence) -> grounded answer
    """
    # 1+2. Hybrid retrieval, reranked
    ranked = llm_rerank(query, candidates=hybrid_search(query, top_n=top_n), keep=keep)

    # 3. Refusal gate: the reranker's top score IS the confidence signal
    top_score = ranked[0].metadata.get("rerank_score", 0.0)
    if top_score < REFUSAL_THRESHOLD:
        print(f"   🚫 REFUSED — top rerank score {top_score:.0f}/10 < {REFUSAL_THRESHOLD:.0f}")
        return (f"I could not find a reliable answer to this in the source "
                f"(best candidate scored {top_score:.0f}/10).")

    # Build numbered, timestamped context from the reranked chunks
    context = []
    for i, node in enumerate(ranked, 1):
        ts = node.metadata.get("approx_timestamp", "?")
        context.append(f"[{i}] (video @{ts})\n{node.text}")
    numbered_context = "\n\n".join(context)

    prompt = (
        "You are a precise RAG assistant. Answer the question using ONLY the "
        "numbered context below. Rules:\n"
        "1. Cite your sources inline as [1], [2], etc. — every claim must have a citation.\n"
        "2. Do NOT use outside knowledge. If the context does not contain the answer, "
        "say exactly: 'Not covered in the source.'\n"
        "3. Keep the answer concise (3-6 sentences).\n\n"
        f"CONTEXT:\n{numbered_context}\n\n"
        f"QUESTION: {query}\n\nANSWER:"
    )

    resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
    answer = _strip_think(resp.message.content).strip()

    # Source list for the user (the "sources" the course talks about)
    sources = [f"[{i}] video @{n.metadata.get('approx_timestamp','?')}"
               for i, n in enumerate(ranked, 1)]
    return answer + "\n\nSOURCES:\n" + "\n".join(sources)

# Two real questions + one the transcript does NOT contain (refusal test)
CITATION_QUERIES = TEST_QUERIES + ["What is the capital of Australia?"]

for q in CITATION_QUERIES:
    print("=" * 72)
    print(f"❓ {q}")
    print("-" * 72)
    print(answer_with_citations(q))
    print()

❓ How does hybrid search combine vector and keyword results?
------------------------------------------------------------------------
   ⏱️  LLM rerank: 4.8s over 10 candidates
Hybrid search combines vector search (semantic) and BM25 (keyword) results using **Reciprocal Rank Fusion (RRF)**, which passes both result sets through a ranking model to produce a final ordered list [1]. Documents that rank well in both searches bubble to the top, while a document ranked #1 in BM25 but absent from vector results may lose to one ranked #3 in both [1]. The two retrievers are combined with adjustable weights (recommended starting at 50/50) to balance semantic relevance with keyword precision [1][2]. The fusion math means the final results are not a simple merge of the two lists [2][3]. In practice, this is implemented by passing the vector retriever and BM25 retriever through an ensemble retriever (or a custom RRF function) to produce the final ranked output [1][3].

SOURCES:
[1] video @02:00:35


## 13. Citation Styles — Inline Source Labels, Multi-Aspect Topics
Section 12 used `[1]` markers + a source list at the bottom. Two upgrades here:
1. **Portable citation labels** — `citation_label()` reads node metadata: transcript chunks → `video @01:56:45`, PDF chunks → `p. 12`. Same function works for both corpora.
2. **Citations woven into the explanation** — for a topic covered by *several* chunks, the answer is organized by sub-aspect and each aspect cites its own source *inline* (no bottom list). The LLM must attribute each claim to the chunk that actually contains it.

In [19]:
def citation_label(node):
    """Portable citation label from node metadata.

    Transcript chunks carry approx_timestamp -> 'video @01:56:45'
    PDF chunks would carry page_number        -> 'p. 12'
    Same function, different corpora — no chunk text modification needed.
    """
    md = node.metadata
    if "approx_timestamp" in md:
        return f"video @{md['approx_timestamp']}"
    if "page_number" in md:
        return f"p. {md['page_number']}"
    return md.get("source", "unknown source")

def answer_woven(query, top_n=RERANK_TOP_N, keep=TOP_K):
    """Grounded answer with source labels woven into the explanation.

    For topics covered by several chunks, the answer is organized by
    sub-aspect and each aspect cites the chunk that actually contains it.
    """
    ranked = llm_rerank(query, candidates=hybrid_search(query, top_n=top_n), keep=keep)

    top_score = ranked[0].metadata.get("rerank_score", 0.0)
    if top_score < REFUSAL_THRESHOLD:
        print(f"   🚫 REFUSED — top rerank score {top_score:.0f}/10 < {REFUSAL_THRESHOLD:.0f}")
        return (f"I could not find a reliable answer to this in the source "
                f"(best candidate scored {top_score:.0f}/10).")

    # Numbered context, each chunk tagged with its portable citation label
    context = []
    for i, node in enumerate(ranked, 1):
        context.append(f"[{i}] ({citation_label(node)})\n{node.text}")
    numbered_context = "\n\n".join(context)

    prompt = (
        "You are a precise RAG assistant. Answer using ONLY the numbered context below.\n"
        "Rules:\n"
        "1. The question covers a topic that may be explained in SEVERAL different chunks. "
        "Organize your answer by sub-aspect (one short paragraph or bullet per aspect).\n"
        "2. After EACH aspect, cite the source that contains it, inline, using its label "
        "in parentheses — e.g. '...exact matching is required (video @01:56:45).' "
        "Do NOT collect citations in a list at the end.\n"
        "3. Attribute each claim to the chunk that actually contains it — do not merge "
        "claims from different chunks under one citation.\n"
        "4. No outside knowledge. If an aspect is not covered, say 'Not covered in the source.'\n\n"
        f"CONTEXT:\n{numbered_context}\n\n"
        f"QUESTION: {query}\n\nANSWER:"
    )

    resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
    return _strip_think(resp.message.content).strip()

# A topic deliberately spread across several chunks (different failure cases,
# different timestamps) — the multi-aspect citation test
WOVEN_QUERY = "What are the main RAG failure modes?"

print("=" * 72)
print(f"❓ {WOVEN_QUERY}")
print("-" * 72)
print(answer_woven(WOVEN_QUERY))

❓ What are the main RAG failure modes?
------------------------------------------------------------------------
   ⏱️  LLM rerank: 1.0s over 10 candidates
Based on the provided context, here are the main RAG failure modes:

**Product codes / SKUs (e.g., "sq7742x"):** The embedding model treats these as meaningless strings with no semantic meaning—just characters—so vector search returns documents about specifications and products in general but misses the specific document containing that code (video @01:56:45).

**Acronyms and abbreviations (e.g., "WCAG 2.1"):** The embedding model does not know what the abbreviation stands for (e.g., "Web Content Accessibility Guidelines"), so a query like "WCA compliance requirements" retrieves documents about compliance and requirements generally but misses the specific WCAG document (video @01:56:45).

**Error codes (e.g., "econ refuse"):** These are literal, non-semantic strings that require exact matching. The embedding model has no idea what th